Import libraries and run needed files

In [11]:
import os
import requests
%run code_breaking_ciphertexts.ipynb
# runs code_breaking_ciphertexts.ipynb so initiates all variables from it to be used here

Fetch moby from gutenberg and write it in a txt. ENSURE `project_dir` matches your own

In [12]:
project_dir = "../.."          # Project/ folder, relative to this notebook
# project_dir = "C:/Users/Jun/2026-27 Mathematical programming/Project"     THIS IS AN EXAMPLE

moby_path = f"{project_dir}/moby.txt"

# download only if the file is missing or empty, so the notebook doesn't hit Gutenberg on every run
if not os.path.exists(moby_path) or os.path.getsize(moby_path) == 0:
    url = "http://www.gutenberg.org/files/2701/2701-0.txt"
    response = requests.get(url)                                   # fetches text from server
    response.raise_for_status()                                    # raises error if server returns some error code e.g. 404
    text = response.content.decode("utf-8-sig", errors="replace")  # decode bytes as utf-8 as requests sometimes gets encoding wrong. replaces undecodable  bytes instead of crashing
    with open(moby_path, "w", encoding="utf-8", newline="") as f:  # open moby_path and write with utf-8 encoding
        f.write(text)                                              # newline="" keeps Gutenberg's \r\n as is

with open(moby_path, "r", encoding="utf-8-sig", errors="replace") as f:
    moby = f.read() # read moby

Trim text so that it starts after *** START and ends before *** END. (Optional)

In [13]:
start_marker_idx = moby.find("*** START OF THE PROJECT GUTENBERG EBOOK")
start_idx = moby.find("\n", start_marker_idx)
end_idx = moby.find("*** END OF THE PROJECT GUTENBERG EBOOK")

if start_idx == -1 or end_idx == -1 or start_marker_idx == -1:
    raise IndexError("start_idx or end_idx is -1")

trimmed_moby = moby[start_idx:end_idx]

Following cell does:
 - counts total characters
 - counts spaces
 - counts total minus spaces
 - counts occurence of each letter
 - creates a list of percentage for each character

In [14]:
# https://www.instructables.com/Break-a-Substitution-Cipher/

valid = ['A','B','C','D','E','F','G','H','I','J','K','L','M','N','O','P','Q','R','S','T','U','V','W','X','Y','Z']
symbols = valid + [" "]

# cleaning: keep A-Z, turn everything else (punctuation, digits, newlines, accents) into a space
cleaned = "".join(letter if letter in valid else " " for letter in trimmed_moby.upper())
cleaned = " ".join(cleaned.split())

total_count = len(cleaned)
print(f"total count: {total_count}")
print(f"space count: {cleaned.count(" ")}")

moby_symbol_counts = []
for symbol in symbols:
    moby_symbol_counts.append([symbol, cleaned.count(symbol)])  # [["A", count], ["B", count], ..., [" ", count]]
assert sum(count for _, count in moby_symbol_counts) == total_count  

moby_symbol_per = []
for char, count in moby_symbol_counts:
    moby_symbol_per.append([char, count/total_count * 100])  # [["A", percent], ["B", percent], ..., " ", percent]]

total count: 1174222
space count: 219053


In [15]:
by_moby_symbol_per = sorted(moby_symbol_per, key=lambda pair: pair[1], reverse=True) # sort letter_per by percentage

print("Freq of symbols:\n")
for symbol, per in by_moby_symbol_per:
    print(f"{symbol!r}: {per:.3f}%") # prints letter sorted by frequency

Freq of symbols:

' ': 18.655%
'E': 10.007%
'T': 7.527%
'A': 6.662%
'O': 5.914%
'N': 5.600%
'I': 5.585%
'S': 5.484%
'H': 5.385%
'R': 4.464%
'L': 3.651%
'D': 3.263%
'U': 2.274%
'M': 1.986%
'C': 1.935%
'W': 1.896%
'F': 1.778%
'G': 1.777%
'P': 1.485%
'B': 1.442%
'Y': 1.439%
'V': 0.734%
'K': 0.689%
'Q': 0.134%
'J': 0.092%
'X': 0.088%
'Z': 0.054%


Applying same method above to `ciphertexts[2]`

In [16]:
# if first line fails, replace ciphertexts[2] with the a copied-pasted version of it
 
cipher2 = ciphertexts[2]

cipher2_total_count = len(cipher2)

cipher2_symbol_count = []
for symbol in symbols:
    cipher2_symbol_count.append([symbol, cipher2.count(symbol)])
assert sum(count for _, count in cipher2_symbol_count) == cipher2_total_count

cipher2_symbol_per = []
for char, count in cipher2_symbol_count:
    cipher2_symbol_per.append([char, count/cipher2_total_count * 100])

In [17]:
by_cipher2_symbol_per = sorted(cipher2_symbol_per, key=lambda pair: pair[1], reverse=True) # sort letter by fequency
by_cipher2_symbol_per

[['W', 18.80781089414183],
 ['C', 9.301130524152107],
 ['T', 7.2456320657759505],
 ['L', 6.783144912641316],
 [' ', 6.423432682425488],
 ['Z', 5.960945529290853],
 ['R', 5.498458376156218],
 ['E', 5.0359712230215825],
 ['A', 4.984583761562178],
 ['F', 4.522096608427543],
 ['V', 3.597122302158273],
 ['Y', 3.545734840698869],
 ['H', 2.4152106885919835],
 ['J', 2.3124357656731758],
 ['K', 2.2610483042137717],
 ['S', 2.055498458376156],
 ['N', 2.004110996916752],
 ['Q', 1.7985611510791366],
 ['B', 1.4388489208633095],
 ['M', 1.3360739979445015],
 ['G', 0.9249743062692704],
 ['U', 0.6680369989722508],
 ['I', 0.6166495375128468],
 ['P', 0.20554984583761562],
 ['D', 0.1541623843782117],
 ['X', 0.10277492291880781],
 ['O', 0.0]]

In [ ]:
conversion_key = {
    cipher_pair[0]: moby_pair[0] for cipher_pair, moby_pair in zip(by_cipher2_symbol_per, by_moby_symbol_per)
} # create dictionary for conversion

In [ ]:
table = str.maketrans(conversion_key) # creates translation table using conversion_key
decrypted2 = cipher2.translate(table) # applies translation to cipher2
print(decrypted2) # "decrypted message"

ISERLOCV SOLMEI GREIERKED SNI CALM GROWEIINOHAL MAHHER FHTNL OFR KNINTOR SAD LEWT FI ALTSOFPS NT UAI EAIB WOR ME USO VHEU SNM IO UELL TO IEE TSAT SE UAI GROWOFHDLB EQCNTED TSE MOMEHT TSAT SNLTOH CFYNTT I YROAD YACV SAD DNIAGGEARED TSROFPS TSE DOOR MB COMRADE RFISED TO TSE TAYLE LAND OFT ALL TSE ILNGI OW GAGER COHTANHNHP DAHCNHP MEH NH WROHT OW SNM AHD TSREU SNMIELW NHTO AH NHTRNCATE AHD ELAYORATE CALCFLATNOH WOR TUO SOFRI N UATCSED SNM AI SE COKERED ISEET AWTER ISEET OW GAGER UNTS WNPFREI AHD LETTERI IO COMGLETELB AYIORYED NH SNI TAIV TSAT SE SAD EKNDEHTLB WORPOTTEH MB GREIEHCE IOMETNMEI SE UAI MAVNHP GROPREII AHD USNITLED AHD IAHP AT SNI UORV IOMETNMEI SE UAI GFJJLED AHD UOFLD INT WOR LOHP IGELLI UNTS A WFRROUED YROU AHD A KACAHT EBE WNHALLB SE IGRAHP WROM SNI CSANR UNTS A CRB OW IATNIWACTNOH AHD UALVED FG AHD DOUH TSE ROOM RFYYNHP SNI SAHDI TOPETSER TSEH SE UROTE A LOHP TELEPRAM FGOH A CAYLE WORM NW MB AHIUER TO TSNI NI AI N SOGE BOF UNLL SAKE A KERB GRETTB CAIE TO ADD TO BOFR COLLEC